In [ ]:
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt

# --- Parameters ---
k = 3.2e-13      # m^-1.5 * h^2.5
N = 30000.0      # h^-1
G = 30.0         # m^3/h
CL_star = 6.0    # mg/L
Vm = 16.0        # g/(L*h)
Ks = 7.0         # g/L
KCL = 0.3        # mg/L
Y_OS = 5.0       # mg/g
Y_PS = 1.5       # g/g

# --- Calculate kLa ---
# kLa = k * N^3 * sqrt(G)
kla = k * (N**3) * np.sqrt(G)
print(f"Calculated kLa: {kla:.2f} h^-1")

# --- Model Equations ---
def reactor_model(y, t):
    S, CL, P = y
    
    # Reaction rate rs (g/(L*h))
    rs = Vm * (S / (Ks + S)) * (CL / (KCL + CL))
    
    # Mass Balances
    dSdt = -rs
    dCLdt = kla * (CL_star - CL) - Y_OS * rs
    dPdt = Y_PS * rs
    
    return [dSdt, dCLdt, dPdt]

# --- Initial Conditions ---
# S0 = 100 g/L, CL0 = 6 mg/L (saturated), P0 = 0 g/L
y0 = [100.0, 6.0, 0.0]
t = np.linspace(0, 10, 500) # Simulate for 10 hours

# --- Solve ODEs ---
sol = odeint(reactor_model, y0, t)
S_res, CL_res, P_res = sol[:, 0], sol[:, 1], sol[:, 2]

# --- Plotting ---
fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot Substrate and Product on left Y-axis
ax1.plot(t, S_res, 'b-', label='Substrate (S)')
ax1.plot(t, P_res, 'g-', label='Product (P)')
ax1.set_xlabel('Time (h)')
ax1.set_ylabel('Concentration (g/L)', color='k')
ax1.grid(alpha=0.3)
ax1.legend(loc='upper left')

# Plot Dissolved Oxygen on right Y-axis
ax2 = ax1.twinx()
ax2.plot(t, CL_res, 'r--', label='Dissolved Oxygen (CL)')
ax2.axhline(KCL, color='orange', linestyle=':', label='K_CL (Limit)')
ax2.set_ylabel('Dissolved Oxygen (mg/L)', color='r')
ax2.tick_params(axis='y', labelcolor='r')
ax2.legend(loc='center right')

plt.title(f'Batch Reactor Simulation (G={G} m³/h, kLa={kla:.2f} h⁻¹)')
plt.show()